In [0]:
file_path = "/Volumes/week4_databricks/default/orders/orders (1).csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)

display(df)

order_id,customer_id,customer_name,city,amount,status
1001,C001,Aarav,Chennai,750,Completed
1002,C002,Diya,Bengaluru,450,Pending
1003,C003,Rahul,Hyderabad,980,Completed
1004,C004,Sneha,Mumbai,1200,Completed
1005,C005,Kiran,Delhi,300,Cancelled
1006,C006,Ananya,Pune,640,Completed
1007,C007,Vikram,Kochi,890,Pending
1008,C008,Meera,Coimbatore,1500,Completed
1009,C009,Arjun,Madurai,520,Completed
1010,C010,Nisha,Trichy,410,Pending


In [0]:
df.printSchema()
df.show()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- status: string (nullable = true)

+--------+-----------+-------------+----------+------+---------+
|order_id|customer_id|customer_name|      city|amount|   status|
+--------+-----------+-------------+----------+------+---------+
|    1001|       C001|        Aarav|   Chennai|   750|Completed|
|    1002|       C002|         Diya| Bengaluru|   450|  Pending|
|    1003|       C003|        Rahul| Hyderabad|   980|Completed|
|    1004|       C004|        Sneha|    Mumbai|  1200|Completed|
|    1005|       C005|        Kiran|     Delhi|   300|Cancelled|
|    1006|       C006|       Ananya|      Pune|   640|Completed|
|    1007|       C007|       Vikram|     Kochi|   890|  Pending|
|    1008|       C008|        Meera|Coimbatore|  1500|Completed|
|    1009|       C009|        Arjun| 

In [0]:
df = df.dropDuplicates()

In [0]:
df = df.dropna()

In [0]:
filtered_df = df.filter(df.amount > 500)

display(filtered_df)

order_id,customer_id,customer_name,city,amount,status
1006,C006,Ananya,Pune,640,Completed
1009,C009,Arjun,Madurai,520,Completed
1007,C007,Vikram,Kochi,890,Pending
1008,C008,Meera,Coimbatore,1500,Completed
1004,C004,Sneha,Mumbai,1200,Completed
1003,C003,Rahul,Hyderabad,980,Completed
1001,C001,Aarav,Chennai,750,Completed


In [0]:
filtered_df.write.mode("overwrite").format("delta").save("/Volumes/week4_databricks/default/orders/clean_orders_delta")

In [0]:
filtered_df.write.mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/week4_databricks/default/orders/clean_orders_csv")

In [0]:
from pyspark.sql.functions import avg, max, sum

filtered_df.count()

filtered_df.select(avg("amount")).show()

filtered_df.select(max("amount")).show()

filtered_df.groupBy("customer_id") \
    .agg(sum("amount").alias("TotalSpent")) \
    .show()

+-----------------+
|      avg(amount)|
+-----------------+
|925.7142857142857|
+-----------------+

+-----------+
|max(amount)|
+-----------+
|       1500|
+-----------+

+-----------+----------+
|customer_id|TotalSpent|
+-----------+----------+
|       C006|       640|
|       C009|       520|
|       C007|       890|
|       C008|      1500|
|       C004|      1200|
|       C003|       980|
|       C001|       750|
+-----------+----------+



In [0]:
filtered_df.createOrReplaceTempView("orders")

In [0]:
%sql
SELECT * FROM orders;

order_id,customer_id,customer_name,city,amount,status
1006,C006,Ananya,Pune,640,Completed
1009,C009,Arjun,Madurai,520,Completed
1007,C007,Vikram,Kochi,890,Pending
1008,C008,Meera,Coimbatore,1500,Completed
1004,C004,Sneha,Mumbai,1200,Completed
1003,C003,Rahul,Hyderabad,980,Completed
1001,C001,Aarav,Chennai,750,Completed


In [0]:
%sql
SELECT COUNT(*) AS TotalOrders
FROM orders;

TotalOrders
7


In [0]:
%sql
SELECT COUNT(*) AS TotalOrders
FROM orders;

TotalOrders
7


In [0]:
%sql
SELECT AVG(amount) AS AverageAmount
FROM orders;

AverageAmount
925.7142857142857


In [0]:
%sql
SELECT MAX(amount) AS HighestAmount
FROM orders;

HighestAmount
1500


In [0]:
%sql
SELECT customer_id,
       SUM(amount) AS TotalSales
FROM orders
GROUP BY customer_id;

customer_id,TotalSales
C006,640
C009,520
C007,890
C008,1500
C004,1200
C003,980
C001,750
